In [ ]:
# =============================================================================
# Colab / Jupyter — install line mirrors `pyre_env/pyproject.toml`
# =============================================================================
#
# [project] dependencies (lines 16–24):
#   openenv-core[core]>=0.2.2, fastapi>=0.115.0, pydantic>=2.0.0,
#   uvicorn[standard]>=0.24.0, requests>=2.31.0, langchain>=1.2.15,
#   langchain-openai>=1.2.1
#
# [project.optional-dependencies] train (lines 31–39), minus Colab-hostile bits:
#   datasets>=4.8.4, peft>=0.15.2, tensorboard>=2.20.0, torch>=2.9.0,
#   transformers>=4.57.6, trl>=1.2.0, tornado>=6.5.5
#   NOT listed here: vllm, flashinfer-*, flash-attn (heavy / GPU-specific; only
#   needed for local GRPO+Unsloth — this PPO notebook uses HTTP + custom torch PPO).
#
# accelerate: not pinned in pyproject but pulled in practice with transformers/TRL.
# trackio: optional for GRPOTrainer(report_to="trackio"); omit if unused.
#
# This PPO notebook at runtime only needs: torch, numpy, matplotlib, requests,
# pydantic. The rest keeps the kernel aligned with the repo for judges / GRPO work.
# =============================================================================

# One line — avoids shell line-continuation quirks on Windows Jupyter.
!pip install -Uq "openenv-core[core]>=0.2.2" "fastapi>=0.115.0" "pydantic>=2.0.0" "uvicorn[standard]>=0.24.0" "requests>=2.31.0" "langchain>=1.2.15" "langchain-openai>=1.2.1" "datasets>=4.8.4" "peft>=0.15.2" "tensorboard>=2.20.0" "transformers>=4.57.6" "trl>=1.2.0" "tornado>=6.5.5" "accelerate"

# Optional — uncomment only if you need GRPO logging to Trackio or local vLLM:
# !pip install -Uq "trackio"
# !pip install -Uq "vllm>=0.11.0,<=0.18.0"

print("✓ pip installs finished (restart runtime if pip upgraded torch).")

# Pyre — PPO Fire-Evacuation RL Training
### OpenEnv Hackathon · Apr 2026

**Pyre** is a partial-observability crisis navigation environment built on [OpenEnv](https://github.com/meta-pytorch/OpenEnv). An RL agent is placed *inside a burning building* and must evacuate alive — navigating fire that spreads each step, smoke that limits visibility, and doors it can close to slow the flames.

This notebook trains a **PPO (Proximal Policy Optimization)** agent on Pyre using a custom actor-critic network with frame-stacking, action masking, GAE, and a patience-gated curriculum (Easy → Medium → Hard).

**Why Pyre?**
- First-person partial observability — the agent sees only nearby cells, not the full map
- Dynamic fire/smoke simulation via cellular automaton — every episode is unique
- 14-component composite reward (movement, danger, strategic door use, survival, speed)
- 3 difficulty levels with adaptive curriculum

| Section | What it does |
|---|---|
| 1 · Imports & Config | All hyper-parameters in one place — edit here only |
| 2 · PPO Utilities | Network, encoder, buffer, GAE, PPO update, curriculum |
| 3 · Connect to Server | Health-check the Pyre HF Space |
| 4 · Build Network | ActorCritic (23K-dim input → 512→256→128 MLP) |
| 5 · Training Loop | PPO with patience curriculum + live charts |
| 6 · Save Artefacts | Model `.pt`, episode CSV, eval CSV |
| 7 · Training Graph | Reward + success-rate over time (PNG) |
| 8 · Per-Difficulty Eval | Success & reward per difficulty |
| 9 · Improvement Chart | First-vs-last eval delta |
| 10 · Health & Steps | HP drain and survival timeline |
| 11 · Summary | Final metrics + saved file list |

---
> **Environment server:** This notebook connects to Pyre running on Hugging Face Spaces at [`Krooz/pyre_env`](https://huggingface.co/spaces/Krooz/pyre_env).
> No local server setup needed — just run the cells top to bottom.

## 1 · Imports & Configuration

Change these values before running — everything else adapts automatically.

In [14]:
import sys, os, time, csv, warnings
from pathlib import Path
from collections import deque
from IPython.display import display, Image as IPImage, clear_output

import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# ── Resolve project root (works whether notebook is run from repo root or examples/) ──
# Walk up from CWD until we find the pyre_env root (identified by openenv.yaml).
# This works whether Jupyter/Colab starts from pyre_env/, training/ppo/, or anywhere else.
_HERE = Path('').resolve()
_ROOT = _HERE
for _ in range(6):
    if (_ROOT / 'openenv.yaml').exists() and (_ROOT / 'server').exists():
        break
    _ROOT = _ROOT.parent
else:
    _ROOT = _HERE  # fallback: no sentinel found, use CWD

for _p in [str(_ROOT), str(_ROOT / 'training' / 'ppo')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f'Project root : {_ROOT}')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device       : {DEVICE}')

Project root : D:\meta hackthon\openenv-pyre
PyTorch      : 2.11.0+cpu
CUDA         : False
Device       : cpu


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING CONFIGURATION  —  edit these values
# ─────────────────────────────────────────────────────────────────────────────

SERVER_URL     = 'https://krooz-pyre-env.hf.space'  # Hugging Face Space server

# ── Scale ────────────────────────────────────────────────────────────────────
EPISODES       = 300          # total training episodes
HISTORY_LENGTH = 4            # frames stacked per observation
HIDDEN_SIZES   = (512, 256, 128)  # MLP hidden layer sizes
OBS_MODE       = 'visible'    # 'visible' = partial obs  |  'full' = oracle

# ── Curriculum (patience-gated) ──────────────────────────────────────────────
CURRICULUM_STAGES  = ['easy', 'medium', 'hard']
PATIENCE_THRESHOLD = 0.55     # success-rate required (30-ep window) to advance
PATIENCE_WINDOW    = 12       # consecutive episodes at threshold before advancing
HARD_MIX_RATIO     = 0.25     # fraction of hard episodes replayed on medium

# ── PPO hyper-parameters ─────────────────────────────────────────────────────
LEARNING_RATE  = 3e-4
LR_END_FACTOR  = 0.1          # final LR = LEARNING_RATE * LR_END_FACTOR
CLIP_EPS       = 0.2
ENTROPY_COEF   = 0.03
VALUE_COEF     = 0.5
GAMMA          = 0.99
GAE_LAMBDA     = 0.95
MAX_GRAD_NORM  = 0.5
UPDATE_EVERY   = 5            # episodes between PPO gradient updates
UPDATE_EPOCHS  = 4
MINIBATCH_SIZE = 256

# ── Evaluation ───────────────────────────────────────────────────────────────
EVAL_EVERY     = 50           # evaluate every N training episodes
EVAL_EPISODES  = 10           # deterministic eval episodes per difficulty

# ── Output ───────────────────────────────────────────────────────────────────
ARTIFACTS_DIR  = _ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH     = ARTIFACTS_DIR / 'pyre_ppo_nb.pt'
CKPT_PATH      = ARTIFACTS_DIR / 'pyre_ppo_nb_ckpt.pt'
CKPT_EVERY     = 50

# ── Resume (set to None to start fresh) ──────────────────────────────────────
RESUME_FROM    = None         # e.g. 'artifacts/pyre_ppo_nb_ckpt.pt'

# ── Visualisation during training ────────────────────────────────────────────
STEP_DELAY_SEC = 0.0          # seconds to sleep between actions (0 = fastest, 0.5 = watchable)
LIVE_PLOT_EVERY = 25          # redraw live training chart every N episodes (0 = disable)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print('Config ready ✓')

Config ready ✓


## 2 · PPO Utilities — Self-Contained

All PPO building blocks are defined **inline** in this cell:
action space, observation encoder, ActorCritic network, rollout buffer, GAE,
PPO update, patience curriculum, training graph helper, HTTP environment wrapper,
and the episode runner. No external script imports needed.

In [24]:
import re
import requests
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
import torch.nn as nn
import torch.nn.functional as F
from pydantic import BaseModel, Field as PydanticField

# ── Standalone Pyre data-model classes (no openenv/openenv_pyre dependency) ───
class PyreMapState(BaseModel):
    """Full numerical grid snapshot for one timestep."""
    grid_w: int; grid_h: int
    template_name: str = ""; episode_id: str = ""; step_count: int = 0; max_steps: int = 150
    cell_grid:  List[int]   = PydanticField(default_factory=list)
    fire_grid:  List[float] = PydanticField(default_factory=list)
    smoke_grid: List[float] = PydanticField(default_factory=list)
    agent_x: int = 0; agent_y: int = 0
    agent_alive: bool = True; agent_evacuated: bool = False; agent_health: float = 100.0
    visible_cells:  List[List[int]]       = PydanticField(default_factory=list)
    exit_positions: List[List[int]]       = PydanticField(default_factory=list)
    door_registry:  Dict[str, List[int]]  = PydanticField(default_factory=dict)
    fire_spread_rate: float = 0.25; wind_dir: str = "CALM"; humidity: float = 0.25


class PyreAction(BaseModel):
    """Agent action (move / look / wait / door)."""
    action:     str
    direction:  Optional[str] = None
    target_id:  Optional[str] = None
    door_state: Optional[str] = None


class PyreObservation(BaseModel):
    """Observation returned by the Pyre env each step (includes reward/done/metadata)."""
    narrative:             str   = ""
    agent_evacuated:       bool  = False
    location_label:        str   = ""
    smoke_level:           str   = "none"
    fire_visible:          bool  = False
    fire_direction:        Optional[str]  = None
    agent_health:          float = 100.0
    health_status:         str   = "Good"
    wind_dir:              str   = "CALM"
    visible_objects:       List[Dict[str, Any]] = PydanticField(default_factory=list)
    blocked_exit_ids:      List[str]            = PydanticField(default_factory=list)
    audible_signals:       List[str]            = PydanticField(default_factory=list)
    elapsed_steps:         int   = 0
    last_action_feedback:  str   = ""
    available_actions_hint: List[str]           = PydanticField(default_factory=list)
    map_state:             Optional[PyreMapState] = None
    reward:                float = 0.0
    done:                  bool  = False
    metadata:              Dict[str, Any]        = PydanticField(default_factory=dict)

# ─────────────────────────────────────────────────────────────────────────────
#  ACTION SPACE
# ─────────────────────────────────────────────────────────────────────────────
MAX_GRID_W   = 24
MAX_GRID_H   = 24
MAX_DOORS    = 16
DIRECTIONS   = ("north", "south", "west", "east")
WINDS        = ("CALM", "NORTH", "SOUTH", "WEST", "EAST")
DIFFICULTIES = ("easy", "medium", "hard")

MOVE_KEYS  = [f"move(direction='{d}')" for d in DIRECTIONS]
LOOK_KEYS  = [f"look(direction='{d}')" for d in DIRECTIONS]
WAIT_KEY   = "wait()"
OPEN_KEYS  = [f"door(target_id='door_{i}', door_state='open')"  for i in range(1, MAX_DOORS + 1)]
CLOSE_KEYS = [f"door(target_id='door_{i}', door_state='close')" for i in range(1, MAX_DOORS + 1)]
ACTION_KEYS     = MOVE_KEYS + LOOK_KEYS + [WAIT_KEY] + OPEN_KEYS + CLOSE_KEYS
ACTION_DIM      = len(ACTION_KEYS)   # 41
ACTION_TO_INDEX = {key: idx for idx, key in enumerate(ACTION_KEYS)}

_MOVE_RE = re.compile(r"move\(direction='(north|south|west|east)'\)")
_LOOK_RE = re.compile(r"look\(direction='(north|south|west|east)'\)")
_DOOR_RE = re.compile(r"door\(target_id='(door_(\d+))', door_state='(open|close)'\)")


def action_index_to_env_action(index: int) -> PyreAction:
    if 0 <= index < 4:
        return PyreAction(action="move", direction=DIRECTIONS[index])
    if 4 <= index < 8:
        return PyreAction(action="look", direction=DIRECTIONS[index - 4])
    if index == 8:
        return PyreAction(action="wait")
    if 9 <= index < 9 + MAX_DOORS:
        return PyreAction(action="door", target_id=f"door_{index - 8}", door_state="open")
    door_slot = index - (9 + MAX_DOORS)
    return PyreAction(action="door", target_id=f"door_{door_slot + 1}", door_state="close")


def build_action_mask(observation: PyreObservation, exclude_look: bool = True) -> np.ndarray:
    mask = np.zeros(ACTION_DIM, dtype=np.float32)
    for hint in observation.available_actions_hint:
        idx = ACTION_TO_INDEX.get(hint)
        if idx is not None:
            if exclude_look and 4 <= idx <= 7:
                continue
            mask[idx] = 1.0
            continue
        m = _MOVE_RE.fullmatch(hint)
        if m:
            mask[ACTION_TO_INDEX[f"move(direction='{m.group(1)}')"]] = 1.0
            continue
        m = _LOOK_RE.fullmatch(hint)
        if m:
            if not exclude_look:
                mask[ACTION_TO_INDEX[f"look(direction='{m.group(1)}')"]] = 1.0
            continue
        m = _DOOR_RE.fullmatch(hint)
        if m:
            door_id, door_num, state = m.group(1), int(m.group(2)), m.group(3)
            if 1 <= door_num <= MAX_DOORS:
                mask[ACTION_TO_INDEX[f"door(target_id='{door_id}', door_state='{state}')"]] = 1.0
    if mask.sum() == 0:
        mask[ACTION_TO_INDEX[WAIT_KEY]] = 1.0
    return mask


# ─────────────────────────────────────────────────────────────────────────────
#  OBSERVATION ENCODER
# ─────────────────────────────────────────────────────────────────────────────
class ObservationEncoder:
    """Encode PyreObservation into a fixed-length float32 vector.

    base_dim = MAX_GRID_W × MAX_GRID_H × 10 channels + 25 global scalars = 5785
    With history stacking of k frames: input_dim = base_dim × k
    """
    base_dim = MAX_GRID_W * MAX_GRID_H * 10 + 25

    def __init__(self, mode: str = "visible"):
        if mode not in {"visible", "full"}:
            raise ValueError(f"mode must be 'visible' or 'full', got '{mode}'")
        self.mode = mode

    def encode(self, observation: PyreObservation) -> np.ndarray:
        ms = observation.map_state
        if ms is None:
            raise ValueError("map_state is required for encoding.")

        cell_one_hot = np.zeros((MAX_GRID_H, MAX_GRID_W, 6), dtype=np.float32)
        fire_ch  = np.zeros((MAX_GRID_H, MAX_GRID_W), dtype=np.float32)
        smoke_ch = np.zeros((MAX_GRID_H, MAX_GRID_W), dtype=np.float32)
        vis_ch   = np.zeros((MAX_GRID_H, MAX_GRID_W), dtype=np.float32)
        agent_ch = np.zeros((MAX_GRID_H, MAX_GRID_W), dtype=np.float32)

        visible = {(x, y) for x, y in ms.visible_cells}
        for y in range(ms.grid_h):
            for x in range(ms.grid_w):
                if self.mode == "visible" and (x, y) not in visible and (x, y) != (ms.agent_x, ms.agent_y):
                    continue
                i = y * ms.grid_w + x
                ct = int(ms.cell_grid[i])
                if 0 <= ct <= 5:
                    cell_one_hot[y, x, ct] = 1.0
                fire_ch[y, x]  = float(ms.fire_grid[i])
                smoke_ch[y, x] = float(ms.smoke_grid[i])
                vis_ch[y, x]   = 1.0 if (x, y) in visible else 0.0

        if 0 <= ms.agent_x < MAX_GRID_W and 0 <= ms.agent_y < MAX_GRID_H:
            agent_ch[ms.agent_y, ms.agent_x] = 1.0

        grid_features = np.concatenate([
            cell_one_hot.reshape(-1), fire_ch.reshape(-1),
            smoke_ch.reshape(-1), vis_ch.reshape(-1), agent_ch.reshape(-1),
        ])

        meta = observation.metadata or {}
        wind = str(meta.get("wind_dir", ms.wind_dir or "CALM")).upper()
        diff = str(meta.get("difficulty", "medium")).lower()
        wi = WINDS.index(wind) if wind in WINDS else 0
        di = DIFFICULTIES.index(diff) if diff in DIFFICULTIES else 1
        wind_oh = np.zeros(len(WINDS), dtype=np.float32);       wind_oh[wi] = 1.0
        diff_oh = np.zeros(len(DIFFICULTIES), dtype=np.float32); diff_oh[di] = 1.0

        # Exit-compass: map-agnostic direction + distance to nearest exit cell
        EXIT_CELL_TYPE = 4
        ax, ay = ms.agent_x, ms.agent_y
        gw, gh = ms.grid_w, ms.grid_h
        best_dist = float(gw + gh)
        best_dx = best_dy = 0.0
        for cy in range(gh):
            for cx in range(gw):
                if int(ms.cell_grid[cy * gw + cx]) == EXIT_CELL_TYPE:
                    d = abs(cx - ax) + abs(cy - ay)
                    if d < best_dist:
                        best_dist = d
                        best_dx = float(cx - ax) / max(1, gw - 1)
                        best_dy = float(cy - ay) / max(1, gh - 1)
        exit_manhattan_norm = best_dist / float(gw + gh)

        global_features = np.array([
            float(observation.agent_health) / 100.0,
            float(ms.agent_health) / 100.0,
            float(ms.step_count) / max(1, ms.max_steps),
            float(ms.fire_spread_rate),
            float(ms.humidity),
            float(ms.agent_x) / max(1, ms.grid_w - 1),
            float(ms.agent_y) / max(1, ms.grid_h - 1),
            float(meta.get("nearest_exit_distance", MAX_GRID_W + MAX_GRID_H) or 0.0) / float(MAX_GRID_W + MAX_GRID_H),
            float(meta.get("reachable_exit_count", 0.0)) / 4.0,
            float(meta.get("visible_cell_count", 0.0)) / float(MAX_GRID_W * MAX_GRID_H),
            float(meta.get("fire_sources", 0.0)) / 5.0,
            {"none": 0.0, "light": 0.33, "moderate": 0.66, "heavy": 1.0}.get(observation.smoke_level, 0.0),
            1.0 if ms.agent_alive else 0.0,
            1.0 if ms.agent_evacuated else 0.0,
            best_dx,
            best_dy,
            exit_manhattan_norm,
        ], dtype=np.float32)

        return np.concatenate([grid_features, global_features, wind_oh, diff_oh]).astype(np.float32)


# ─────────────────────────────────────────────────────────────────────────────
#  ACTOR-CRITIC NETWORK
# ─────────────────────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    """Shared-backbone MLP Actor-Critic for PPO with LayerNorm and orthogonal init."""

    def __init__(self, input_dim: int, action_dim: int, hidden_sizes: Tuple[int, ...] = (512, 256, 128)):
        super().__init__()
        h1, h2, h3 = hidden_sizes
        self.shared = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, h1), nn.LayerNorm(h1), nn.ReLU(),
            nn.Linear(h1, h2),        nn.LayerNorm(h2), nn.ReLU(),
            nn.Linear(h2, h3),                          nn.ReLU(),
        )
        self._init_orthogonal()
        self.policy_head = nn.Linear(h3, action_dim)
        self.value_head  = nn.Linear(h3, 1)
        nn.init.orthogonal_(self.policy_head.weight, gain=0.01)
        nn.init.zeros_(self.policy_head.bias)
        nn.init.orthogonal_(self.value_head.weight, gain=1.0)
        nn.init.zeros_(self.value_head.bias)

    def _init_orthogonal(self):
        for layer in self.shared:
            if isinstance(layer, nn.Linear):
                nn.init.orthogonal_(layer.weight, gain=np.sqrt(2))
                nn.init.zeros_(layer.bias)

    def forward(self, obs: torch.Tensor, mask: torch.Tensor):
        features = self.shared(obs)
        logits = self.policy_head(features)
        logits = torch.where(mask.bool(), logits, torch.full_like(logits, -1e9))
        dist   = torch.distributions.Categorical(logits=logits)
        values = self.value_head(features).squeeze(-1)
        return dist, values

    def act(self, obs, mask, deterministic=False):
        dist, values = self(obs, mask)
        action = dist.mode if deterministic else dist.sample()
        return action, dist.log_prob(action), values

    def evaluate(self, obs, mask, action):
        dist, values = self(obs, mask)
        return dist.log_prob(action), values, dist.entropy()


# ─────────────────────────────────────────────────────────────────────────────
#  ROLLOUT BUFFER
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class RolloutBuffer:
    obs:       List[np.ndarray] = field(default_factory=list)
    masks:     List[np.ndarray] = field(default_factory=list)
    actions:   List[int]        = field(default_factory=list)
    rewards:   List[float]      = field(default_factory=list)
    log_probs: List[float]      = field(default_factory=list)
    values:    List[float]      = field(default_factory=list)
    dones:     List[bool]       = field(default_factory=list)

    def clear(self):
        for lst in (self.obs, self.masks, self.actions,
                    self.rewards, self.log_probs, self.values, self.dones):
            lst.clear()

    def __len__(self):
        return len(self.rewards)


# ─────────────────────────────────────────────────────────────────────────────
#  GAE
# ─────────────────────────────────────────────────────────────────────────────
def compute_gae(rewards, values, dones, gamma, gae_lambda):
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae = next_value = 0.0
    for t in reversed(range(T)):
        if dones[t]:
            next_value = gae = 0.0
        delta = rewards[t] + gamma * next_value * (1.0 - dones[t]) - values[t]
        gae   = delta + gamma * gae_lambda * (1.0 - dones[t]) * gae
        advantages[t] = gae
        next_value = values[t]
    return advantages + values, advantages


# ─────────────────────────────────────────────────────────────────────────────
#  PPO UPDATE
# ─────────────────────────────────────────────────────────────────────────────
def ppo_update(network, optimizer, buffer, device, clip_eps, value_clip_eps,
               entropy_coef, value_coef, n_epochs, minibatch_size,
               gamma, gae_lambda, max_grad_norm) -> Dict[str, float]:
    rewards = np.array(buffer.rewards, dtype=np.float32)
    values  = np.array(buffer.values,  dtype=np.float32)
    dones   = np.array(buffer.dones,   dtype=np.float32)
    returns, advantages = compute_gae(rewards, values, dones, gamma, gae_lambda)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    obs_arr       = torch.tensor(np.stack(buffer.obs),   dtype=torch.float32, device=device)
    mask_arr      = torch.tensor(np.stack(buffer.masks), dtype=torch.float32, device=device)
    action_arr    = torch.tensor(buffer.actions,         dtype=torch.long,    device=device)
    old_logp_arr  = torch.tensor(buffer.log_probs,       dtype=torch.float32, device=device)
    return_arr    = torch.tensor(returns,                dtype=torch.float32, device=device)
    adv_arr       = torch.tensor(advantages,             dtype=torch.float32, device=device)
    old_value_arr = torch.tensor(values,                 dtype=torch.float32, device=device)

    T = len(buffer)
    metrics   = {"policy_loss": 0.0, "value_loss": 0.0, "entropy": 0.0,
                 "approx_kl": 0.0, "clip_frac": 0.0}
    n_updates = 0

    network.train()
    for _ in range(n_epochs):
        perm = torch.randperm(T, device=device)
        for start in range(0, T, minibatch_size):
            idx = perm[start:start + minibatch_size]
            if len(idx) < 2:
                continue
            log_prob, value, entropy = network.evaluate(obs_arr[idx], mask_arr[idx], action_arr[idx])
            ratio   = torch.exp(log_prob - old_logp_arr[idx])
            adv_mb  = adv_arr[idx]
            policy_loss = -torch.min(
                ratio * adv_mb,
                torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv_mb,
            ).mean()
            ret_mb  = return_arr[idx]
            old_val = old_value_arr[idx]
            v_clip  = old_val + torch.clamp(value - old_val, -value_clip_eps, value_clip_eps)
            value_loss = torch.max(F.mse_loss(value, ret_mb), F.mse_loss(v_clip, ret_mb))
            loss = policy_loss + value_coef * value_loss - entropy_coef * entropy.mean()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(network.parameters(), max_grad_norm)
            optimizer.step()
            with torch.no_grad():
                metrics["policy_loss"] += policy_loss.item()
                metrics["value_loss"]  += value_loss.item()
                metrics["entropy"]     += entropy.mean().item()
                metrics["approx_kl"]   += ((ratio - 1) - (log_prob - old_logp_arr[idx])).mean().item()
                metrics["clip_frac"]   += ((ratio - 1.0).abs() > clip_eps).float().mean().item()
            n_updates += 1

    if n_updates > 0:
        for k in metrics:
            metrics[k] /= n_updates
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
#  CURRICULUM
# ─────────────────────────────────────────────────────────────────────────────
def build_curriculum(schedule_str: str, n_episodes: int) -> List[str]:
    stages = [s.strip().lower() for s in schedule_str.split(",") if s.strip()] or ["medium"]
    seg    = max(1, n_episodes // len(stages))
    schedule: List[str] = []
    for s in stages:
        schedule.extend([s] * seg)
    while len(schedule) < n_episodes:
        schedule.append(stages[-1])
    return schedule[:n_episodes]


class PatienceCurriculum:
    def __init__(self, stages, threshold, patience_window, mix_ratio=0.0):
        self.stages = stages; self.threshold = threshold
        self.patience_window = patience_window; self.mix_ratio = mix_ratio
        self.stage_idx = 0; self._streak = 0

    @property
    def current(self) -> str:
        return self.stages[self.stage_idx]

    def step(self, success_rate_30: float) -> str:
        if self.stage_idx < len(self.stages) - 1:
            self._streak = self._streak + 1 if success_rate_30 >= self.threshold else 0
            if self._streak >= self.patience_window:
                self.stage_idx += 1; self._streak = 0
                print(f"  [curriculum] Advanced to '{self.current}' "
                      f"(suc30={success_rate_30:.2f} >= {self.threshold} "
                      f"for {self.patience_window} eps)")
        if self.current == "hard" and self.mix_ratio > 0.0 and len(self.stages) >= 2:
            if np.random.rand() < self.mix_ratio:
                return self.stages[self.stage_idx - 1]
        return self.current


# ─────────────────────────────────────────────────────────────────────────────
#  TRAINING GRAPH
# ─────────────────────────────────────────────────────────────────────────────
def save_training_graph_png(path, episode_rows, eval_rows, window=20):
    if not episode_rows:
        return
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    episodes   = [int(r["episode"])    for r in episode_rows]
    rewards    = [float(r["reward"])   for r in episode_rows]
    evacuated  = [float(r["evacuated"]) for r in episode_rows]
    difficulty = [str(r["difficulty"]) for r in episode_rows]

    def ma(values, w):
        out, run, q = [], 0.0, []
        for v in values:
            q.append(v); run += v
            if len(q) > w: run -= q.pop(0)
            out.append(run / len(q))
        return out

    reward_ma  = ma(rewards, window)
    success_ma = ma(evacuated, window)
    eval_eps   = [int(r["episode"])        for r in eval_rows]
    eval_succ  = [float(r["success_rate"]) for r in eval_rows]

    diff_colors = {"easy": "#d4edda", "medium": "#fff3cd", "hard": "#f8d7da"}
    regions: List[tuple] = []
    if difficulty:
        cur, start = difficulty[0], episodes[0]
        for ep, d in zip(episodes[1:], difficulty[1:]):
            if d != cur:
                regions.append((start, ep, cur)); cur, start = d, ep
        regions.append((start, episodes[-1], cur))

    import matplotlib.patches as mpatches
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()
    for x0, x1, diff in regions:
        ax1.axvspan(x0, x1, color=diff_colors.get(diff, "#eee"), alpha=0.35, zorder=0)
    ax1.axhline(0, color="#aaa", linewidth=0.8, linestyle="--", zorder=1)
    ax1.plot(episodes, rewards,    color="#d1c7bc", linewidth=0.8, alpha=0.6, label="Episode reward", zorder=2)
    ax1.plot(episodes, reward_ma,  color="#c1661c", linewidth=2.5, label=f"Reward (MA-{window})", zorder=3)
    ax2.plot(episodes, success_ma, color="#1a7a8a", linewidth=2.5, label=f"Success rate (MA-{window})", zorder=3)
    if eval_eps:
        ax2.scatter(eval_eps, eval_succ, color="#0d5b6b", s=60, zorder=5,
                    marker="D", label="Eval success", edgecolors="white", linewidths=1.2)
    ax1.set_xlabel("Episode", fontsize=13, fontweight="bold")
    ax1.set_ylabel("Reward",  fontsize=13, fontweight="bold", color="#c1661c")
    ax2.set_ylabel("Success Rate", fontsize=13, fontweight="bold", color="#1a7a8a")
    ax2.set_ylim(-0.05, 1.05)
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax1.grid(True, linestyle="--", linewidth=0.6, color="#ddd", alpha=0.8, zorder=0)
    ax1.set_xlim(episodes[0], episodes[-1])
    diff_patches = [mpatches.Patch(color=diff_colors[d], alpha=0.6, label=d.capitalize())
                    for d in ["easy", "medium", "hard"] if any(r == d for r in difficulty)]
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2 + diff_patches, l1 + l2 + [p.get_label() for p in diff_patches],
               loc="upper left", fontsize=9, framealpha=0.85)
    final_sr = success_ma[-1] if success_ma else 0.0
    fig.suptitle(
        f"Pyre PPO Training  —  {episodes[-1]} episodes  |  final success rate: {final_sr:.0%}",
        fontsize=14, fontweight="bold", y=1.01,
    )
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
#  HTTP ENVIRONMENT WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
class HttpPyreEnv:
    """Thin HTTP client for the Pyre REST API (POST /reset, POST /step)."""

    def __init__(self, base_url: str = "http://localhost:8000", timeout: int = 60):
        self.base_url = base_url.rstrip("/")
        self.timeout  = timeout
        self.session  = requests.Session()
        self.session.headers.update({"Content-Type": "application/json"})

    def _parse(self, data: Dict[str, Any]) -> PyreObservation:
        obs_raw   = data.get("observation", data)
        ms_raw    = obs_raw.get("map_state")
        map_state = PyreMapState(**ms_raw) if ms_raw else None
        return PyreObservation(
            narrative=obs_raw.get("narrative", ""),
            agent_evacuated=obs_raw.get("agent_evacuated", False),
            location_label=obs_raw.get("location_label", ""),
            smoke_level=obs_raw.get("smoke_level", "none"),
            fire_visible=obs_raw.get("fire_visible", False),
            fire_direction=obs_raw.get("fire_direction"),
            agent_health=float(obs_raw.get("agent_health", 100.0)),
            health_status=obs_raw.get("health_status", "Good"),
            wind_dir=obs_raw.get("wind_dir", "CALM"),
            visible_objects=obs_raw.get("visible_objects", []),
            blocked_exit_ids=obs_raw.get("blocked_exit_ids", []),
            audible_signals=obs_raw.get("audible_signals", []),
            elapsed_steps=obs_raw.get("elapsed_steps", 0),
            last_action_feedback=obs_raw.get("last_action_feedback", ""),
            available_actions_hint=obs_raw.get("available_actions_hint", []),
            map_state=map_state,
            reward=float(data.get("reward", 0.0)),
            done=bool(data.get("done", False)),
            metadata=data.get("metadata", {}),
        )

    def _request_with_retry(self, method: str, url: str, **kwargs) -> requests.Response:
        """Perform an HTTP request with exponential backoff for rate limits and 5xx errors."""
        max_retries = 10
        base_delay  = 1.0
        for i in range(max_retries):
            try:
                resp = self.session.request(method, url, **kwargs)
                if resp.status_code == 429:
                    wait = base_delay * (2 ** i) + np.random.uniform(0, 1)
                    print(f"  [http] Rate limited (429). Retrying in {wait:.1f}s... (attempt {i+1}/{max_retries})")
                    time.sleep(wait); continue
                if 500 <= resp.status_code < 600:
                    wait = base_delay * (2 ** i)
                    print(f"  [http] Server error ({resp.status_code}). Retrying in {wait:.1f}s... (attempt {i+1}/{max_retries})")
                    time.sleep(wait); continue
                resp.raise_for_status()
                return resp
            except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
                if i == max_retries - 1: raise
                wait = base_delay * (2 ** i)
                print(f"  [http] Network error: {e}. Retrying in {wait:.1f}s... (attempt {i+1}/{max_retries})")
                time.sleep(wait)
        raise requests.exceptions.RetryError("Max retries exceeded")

    def reset(self, difficulty: str = "easy", seed: Optional[int] = None) -> PyreObservation:
        payload: Dict[str, Any] = {"difficulty": difficulty}
        if seed is not None: payload["seed"] = seed
        resp = self._request_with_retry("POST", f"{self.base_url}/reset", json=payload, timeout=self.timeout)
        return self._parse(resp.json())

    def step(self, action: PyreAction) -> PyreObservation:
        payload: Dict[str, Any] = {"action": action.action}
        if action.direction  is not None: payload["direction"]  = action.direction
        if action.target_id  is not None: payload["target_id"]  = action.target_id
        if action.door_state is not None: payload["door_state"] = action.door_state
        resp = self._request_with_retry("POST", f"{self.base_url}/step", json=payload, timeout=self.timeout)
        return self._parse(resp.json())

    def health_check(self) -> bool:
        try:
            r = self.session.get(f"{self.base_url}/state", timeout=5)
            return r.status_code < 500
        except requests.exceptions.RequestException:
            return False


# ─────────────────────────────────────────────────────────────────────────────
#  EPISODE RESULT + RUNNER
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class EpisodeResult:
    total_reward: float
    steps: int
    evacuated: bool
    final_health: float
    difficulty: str


def run_episode(
    env: HttpPyreEnv,
    network: ActorCritic,
    encoder: ObservationEncoder,
    device: torch.device,
    difficulty: str,
    history_length: int,
    buffer: RolloutBuffer,
    deterministic: bool = False,
    step_delay: float = 0.0,
) -> EpisodeResult:
    observation = env.reset(difficulty=difficulty)
    zero_frame  = np.zeros(encoder.base_dim, dtype=np.float32)
    frames: deque = deque([zero_frame.copy() for _ in range(history_length)], maxlen=history_length)
    frames.append(encoder.encode(observation))

    total_reward     = 0.0
    final_health     = observation.agent_health
    evacuated        = False
    steps            = 0
    LOOP_WINDOW      = 12
    recent_positions: deque = deque(maxlen=LOOP_WINDOW)

    network.eval()
    with torch.no_grad():
        while True:
            state_vec   = np.concatenate(list(frames), dtype=np.float32)
            action_mask = build_action_mask(observation, exclude_look=True)

            obs_t  = torch.tensor(state_vec,   dtype=torch.float32, device=device).unsqueeze(0)
            mask_t = torch.tensor(action_mask, dtype=torch.float32, device=device).unsqueeze(0)

            action_t, log_prob_t, value_t = network.act(obs_t, mask_t, deterministic=deterministic)
            action_idx = int(action_t.item())
            env_action = action_index_to_env_action(action_idx)

            if step_delay > 0.0:
                time.sleep(step_delay)

            next_obs = env.step(env_action)

            reward        = float(next_obs.reward or 0.0)
            chosen_action = env_action.action

            # Shaping 1 — idle penalty
            if chosen_action == "wait":
                reward -= 0.05

            # Shaping 2 — fire-approach penalty
            ms_next = next_obs.map_state
            if ms_next is not None and chosen_action.startswith("move"):
                ax, ay = ms_next.agent_x, ms_next.agent_y
                gw, gh = ms_next.grid_w, ms_next.grid_h
                for dx, dy in ((0, 1), (0, -1), (1, 0), (-1, 0)):
                    nx, ny = ax + dx, ay + dy
                    if 0 <= nx < gw and 0 <= ny < gh:
                        if float(ms_next.fire_grid[ny * gw + nx]) > 0.15:
                            reward -= 0.15
                            break

            # Shaping 3 — anti-loop penalty
            if ms_next is not None and chosen_action.startswith("move"):
                cur_pos = (ms_next.agent_x, ms_next.agent_y)
                if cur_pos in recent_positions:
                    reward -= 0.2
                recent_positions.append(cur_pos)

            # Shaping 4 — exit proximity pull
            if ms_next is not None and chosen_action.startswith("move") and not next_obs.agent_evacuated:
                ax, ay = ms_next.agent_x, ms_next.agent_y
                exits  = ms_next.exit_positions
                if exits:
                    min_manhattan = min(abs(ax - ex[0]) + abs(ay - ex[1]) for ex in exits)
                    reward += max(0.0, 0.25 - 0.04 * min_manhattan)

            done = bool(next_obs.done)

            buffer.obs.append(state_vec)
            buffer.masks.append(action_mask)
            buffer.actions.append(action_idx)
            buffer.rewards.append(reward)
            buffer.log_probs.append(float(log_prob_t.item()))
            buffer.values.append(float(value_t.item()))
            buffer.dones.append(done)

            total_reward += reward
            steps        += 1
            final_health  = next_obs.agent_health
            evacuated     = next_obs.agent_evacuated

            frames.append(encoder.encode(next_obs))
            observation = next_obs
            if done:
                break

    return EpisodeResult(
        total_reward=total_reward, steps=steps,
        evacuated=evacuated, final_health=final_health, difficulty=difficulty,
    )


# ── Initialise shared objects ──────────────────────────────────────────────────
encoder   = ObservationEncoder(mode=OBS_MODE)
input_dim = encoder.base_dim * HISTORY_LENGTH

print(f'Encoder      : base_dim={encoder.base_dim} × {HISTORY_LENGTH} frames → input_dim={input_dim:,}')
print(f'Action space : {ACTION_DIM} discrete actions')
print(f'Difficulties : {DIFFICULTIES}')
print('All PPO utilities loaded (self-contained) ✓')

Encoder      : base_dim=5785 × 4 frames → input_dim=23,140
Action space : 37 discrete actions
Difficulties : ('easy', 'medium', 'hard')
All PPO utilities loaded (self-contained) ✓


## 3 · Connect to Pyre Server

In [25]:
env = HttpPyreEnv(base_url=SERVER_URL)

is_hf = 'hf.space' in SERVER_URL
print(f'Connecting to {SERVER_URL}')
print('(HF Space — first request may take ~30s if the Space is sleeping...)' if is_hf else '')
print('Checking ... ', end='', flush=True)

if env.health_check():
    print('Connected ✓')
else:
    print('FAILED ✗')
    if is_hf:
        raise RuntimeError(
            f'Pyre HF Space not reachable at {SERVER_URL}\n'
            'Check that the Space is running at:\n'
            '  https://huggingface.co/spaces/Krooz/pyre_env'
        )
    else:
        raise RuntimeError(
            f'Pyre server not reachable at {SERVER_URL}\n'
            'Start it first:\n'
            '  cd openenv-pyre\n'
            '  uv run server/app.py'
        )

Connecting to https://akshaykumarbm-pyre-env.hf.space
(HF Space — first request may take ~30s if the Space is sleeping...)
Checking ... Connected ✓


## 4 · Build Network  *(+ optional resume)*

Set `RESUME_FROM` in the config cell to continue a previous run.

In [26]:
device  = torch.device(DEVICE)
network = ActorCritic(
    input_dim=input_dim,
    action_dim=ACTION_DIM,
    hidden_sizes=HIDDEN_SIZES,
).to(device)

optimizer = torch.optim.Adam(network.parameters(), lr=LEARNING_RATE, eps=1e-5)

total_updates = EPISODES // UPDATE_EVERY
lr_scheduler  = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=LR_END_FACTOR,
    total_iters=max(1, total_updates),
)

n_params  = sum(p.numel() for p in network.parameters())
START_EP  = 0

# ── Resume ───────────────────────────────────────────────────────────────────
if RESUME_FROM and Path(RESUME_FROM).exists():
    ckpt = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    network.load_state_dict(ckpt.get('network_state', ckpt))
    if 'optimizer_state' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer_state'])
    if ckpt.get('scheduler_state'):
        lr_scheduler.load_state_dict(ckpt['scheduler_state'])
    START_EP = int(ckpt.get('episode', 0))
    print(f'Resumed from episode {START_EP}: {RESUME_FROM}')
elif RESUME_FROM:
    print(f'Checkpoint not found: {RESUME_FROM} — starting fresh')

print(f'Device       : {device}')
print(f'Parameters   : {n_params:,}')
print(f'Architecture : {HIDDEN_SIZES}')
print(f'Start episode: {START_EP} / {EPISODES}')
print('Network ready ✓')

Device       : cpu
Parameters   : 12,065,650
Architecture : (512, 256, 128)
Start episode: 0 / 300
Network ready ✓


## 5 · Training Loop

**Patience-gated curriculum**: the agent starts on Easy and advances only when
30-episode success rate ≥ `PATIENCE_THRESHOLD` for `PATIENCE_WINDOW` consecutive
episodes.  Set `STEP_DELAY_SEC = 0.5` to watch each action in the browser visualiser.

In [27]:
# ── Curriculum ───────────────────────────────────────────────────────────────
curriculum = PatienceCurriculum(
    stages=CURRICULUM_STAGES,
    threshold=PATIENCE_THRESHOLD,
    patience_window=PATIENCE_WINDOW,
    mix_ratio=HARD_MIX_RATIO,
)

# ── Tracking ─────────────────────────────────────────────────────────────────
buffer         = RolloutBuffer()
episode_rows   = []
eval_rows      = []
reward_window  = deque(maxlen=30)
success_window = deque(maxlen=30)
t_start        = time.time()

# ── Helper: live chart (redrawn every LIVE_PLOT_EVERY episodes) ──────────────
def _live_chart(episode_rows, eval_rows):
    if len(episode_rows) < 2:
        return
    eps    = [r['episode']          for r in episode_rows]
    rews   = [r['reward']           for r in episode_rows]
    suc    = [r['success_rate_30']  for r in episode_rows]
    ev_eps = [r['episode']          for r in eval_rows if r['difficulty'] == 'medium']
    ev_suc = [r['success_rate']     for r in eval_rows if r['difficulty'] == 'medium']

    fig, ax1 = plt.subplots(figsize=(13, 4))
    ax2 = ax1.twinx()
    ax1.plot(eps, rews, color='#d1c7bc', linewidth=0.7, alpha=0.5)
    ax1.plot(eps, [r['reward_mean_30'] for r in episode_rows],
             color='#c1661c', linewidth=2.2, label='Reward (MA-30)')
    ax2.plot(eps, suc, color='#1a7a8a', linewidth=2.2, label='Success (MA-30)')
    if ev_eps:
        ax2.scatter(ev_eps, ev_suc, color='#0d5b6b', s=55, zorder=5,
                    marker='D', edgecolors='white', linewidths=1, label='Eval (medium)')
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward', color='#c1661c')
    ax2.set_ylabel('Success Rate', color='#1a7a8a')
    ax2.set_ylim(-0.05, 1.05)
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8)
    ax1.grid(True, linestyle='--', linewidth=0.5, alpha=0.6)
    fig.suptitle(f'Pyre PPO — Live Training  (ep {eps[-1]} / {EPISODES})',
                 fontsize=11, fontweight='bold')
    fig.tight_layout()
    clear_output(wait=True)
    plt.show()
    plt.close(fig)


print(f'Starting training: episodes {START_EP+1}–{EPISODES} on {device}')
print(f'Curriculum   : {CURRICULUM_STAGES}  threshold={PATIENCE_THRESHOLD}  window={PATIENCE_WINDOW}')
print(f'Step delay   : {STEP_DELAY_SEC}s   (set STEP_DELAY_SEC=0.5 to watch in browser)')
print('-' * 80, flush=True)

for ep_idx in range(START_EP, EPISODES):
    ep         = ep_idx + 1
    difficulty = curriculum.current

    # ── Run one episode against the Pyre HF Space ────────────────────
    result = run_episode(
        env=env, network=network, encoder=encoder, device=device,
        difficulty=difficulty, history_length=HISTORY_LENGTH,
        buffer=buffer, deterministic=False,
        step_delay=STEP_DELAY_SEC,
    )

    reward_window.append(result.total_reward)
    success_window.append(float(result.evacuated))
    r30   = float(np.mean(reward_window))
    suc30 = float(np.mean(success_window))

    difficulty = curriculum.step(suc30)

    elapsed = time.time() - t_start
    episode_rows.append({
        'episode': ep, 'difficulty': result.difficulty,
        'reward': round(result.total_reward, 4),
        'evacuated': int(result.evacuated),
        'steps': result.steps,
        'final_health': round(result.final_health, 2),
        'reward_mean_30': round(r30, 4),
        'success_rate_30': round(suc30, 4),
    })

    print(
        f'ep={ep:04d} [{result.difficulty:<6}] '
        f'steps={result.steps:03d}  reward={result.total_reward:+8.3f}  '
        f'evac={int(result.evacuated)}  hp={result.final_health:5.1f}  '
        f'suc30={suc30:.2f}  r30={r30:+7.2f}  t={elapsed:.0f}s',
        flush=True,
    )

    # ── PPO update every UPDATE_EVERY episodes ────────────────────────
    should_update = (ep % UPDATE_EVERY == 0) or (ep == EPISODES)
    if should_update and len(buffer) > 0:
        network.train()
        ppo_m = ppo_update(
            network=network, optimizer=optimizer, buffer=buffer, device=device,
            clip_eps=CLIP_EPS, value_clip_eps=CLIP_EPS,
            entropy_coef=ENTROPY_COEF, value_coef=VALUE_COEF,
            n_epochs=UPDATE_EPOCHS, minibatch_size=MINIBATCH_SIZE,
            gamma=GAMMA, gae_lambda=GAE_LAMBDA, max_grad_norm=MAX_GRAD_NORM,
        )
        lr_scheduler.step()
        buffer.clear()
        network.eval()
        cur_lr = optimizer.param_groups[0]['lr']
        print(
            f'  >> PPO  pi={ppo_m["policy_loss"]:+.4f}  '
            f'v={ppo_m["value_loss"]:.4f}  '
            f'H={ppo_m["entropy"]:.4f}  '
            f'kl={ppo_m["approx_kl"]:.4f}  '
            f'clip%={ppo_m["clip_frac"]:.2f}  lr={cur_lr:.2e}',
            flush=True,
        )

    # ── Periodic evaluation (all 3 difficulties) ──────────────────────
    if EVAL_EVERY > 0 and (ep % EVAL_EVERY == 0 or ep == EPISODES):
        for eval_diff in DIFFICULTIES:
            ev_r, ev_s, ev_st = [], [], []
            ev_buf = RolloutBuffer()
            for _ in range(EVAL_EPISODES):
                er = run_episode(
                    env=env, network=network, encoder=encoder, device=device,
                    difficulty=eval_diff, history_length=HISTORY_LENGTH,
                    buffer=ev_buf, deterministic=True,
                )
                ev_buf.clear()
                ev_r.append(er.total_reward)
                ev_s.append(float(er.evacuated))
                ev_st.append(er.steps)
            avg_r  = float(np.mean(ev_r))
            avg_s  = float(np.mean(ev_s))
            avg_st = float(np.mean(ev_st))
            eval_rows.append({
                'episode': ep, 'difficulty': eval_diff,
                'reward_mean': round(avg_r, 4),
                'success_rate': round(avg_s, 3),
                'steps_mean': round(avg_st, 1),
            })
            print(
                f'  ** EVAL [{eval_diff:<6}]  '
                f'reward={avg_r:+.3f}  success={avg_s:.2f}  steps={avg_st:.1f}',
                flush=True,
            )

    # ── Periodic checkpoint ───────────────────────────────────────────
    if CKPT_EVERY > 0 and ep % CKPT_EVERY == 0:
        torch.save({
            'episode': ep,
            'network_state': network.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': lr_scheduler.state_dict(),
        }, CKPT_PATH)
        print(f'  [ckpt] saved → {CKPT_PATH}', flush=True)

    # ── Live chart ────────────────────────────────────────────────────
    if LIVE_PLOT_EVERY > 0 and ep % LIVE_PLOT_EVERY == 0:
        _live_chart(episode_rows, eval_rows)

total_time = time.time() - t_start
trained_eps = EPISODES - START_EP
print('=' * 80)
print(f'Training done: {trained_eps} episodes in {total_time:.1f}s  ({trained_eps/max(1,total_time):.2f} eps/s)')
print(f'Final success rate (last 30): {float(np.mean(success_window)):.2f}')
print(f'Final reward mean  (last 30): {float(np.mean(reward_window)):+.3f}')
print('=' * 80, flush=True)

Starting training: episodes 1–300 on cpu
Curriculum   : ['easy', 'medium', 'hard']  threshold=0.55  window=12
Step delay   : 0.0s   (set STEP_DELAY_SEC=0.5 to watch in browser)
--------------------------------------------------------------------------------
    step=001  action=door(target_id='door_7', door_state='open')  hp=100.0
    step=002  action=move(direction='north')                   hp=100.0
    step=003  action=door(target_id='door_8', door_state='close')  hp=100.0
    step=004  action=move(direction='east')                    hp=100.0
ep=0001 [easy  ] steps=004  reward= +16.810  evac=1  hp=100.0  suc30=1.00  r30= +16.81  t=3s
    step=001  action=move(direction='east')                    hp=100.0
    step=002  action=move(direction='north')                   hp=100.0
    step=003  action=move(direction='north')                   hp=100.0
    step=004  action=move(direction='south')                   hp=100.0
    step=005  action=wait()                                    hp=

KeyboardInterrupt: 

## 6 · Save Model & Metrics

In [ ]:
# ── Final model checkpoint ────────────────────────────────────────────────────
torch.save({
    'episode': EPISODES,
    'network_state': network.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'scheduler_state': lr_scheduler.state_dict(),
    'config': {
        'input_dim': input_dim, 'action_dim': ACTION_DIM,
        'hidden_sizes': HIDDEN_SIZES, 'history_length': HISTORY_LENGTH,
        'obs_mode': OBS_MODE,
    },
}, MODEL_PATH)
print(f'Model saved      → {MODEL_PATH}')

# ── Episode CSV ───────────────────────────────────────────────────────────────
if episode_rows:
    ep_csv = MODEL_PATH.with_suffix('.csv')
    with open(ep_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=episode_rows[0].keys())
        w.writeheader(); w.writerows(episode_rows)
    print(f'Episode CSV      → {ep_csv}')

# ── Eval CSV ──────────────────────────────────────────────────────────────────
if eval_rows:
    eval_csv = MODEL_PATH.parent / (MODEL_PATH.stem + '_eval.csv')
    with open(eval_csv, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=eval_rows[0].keys())
        w.writeheader(); w.writerows(eval_rows)
    print(f'Eval CSV         → {eval_csv}')

print('All artefacts saved ✓')

## 7 · Training Graph — Reward & Success Rate Over Time

In [ ]:
GRAPH_PATH = ARTIFACTS_DIR / 'pyre_ppo_nb_training.png'

# Only pass medium eval rows as the scatter overlay so the graph matches train_torch_ppo.py
eval_rows_medium = [r for r in eval_rows if r['difficulty'] == 'medium']
save_training_graph_png(GRAPH_PATH, episode_rows, eval_rows_medium, window=20)

print(f'Training graph saved → {GRAPH_PATH}')
display(IPImage(filename=str(GRAPH_PATH)))

## 8 · Per-Difficulty Evaluation Graph

Shows how **success rate** and **mean reward** evolve on each difficulty throughout
training — reveals when the agent starts generalising beyond Easy.

In [ ]:
def plot_per_difficulty_eval(eval_rows, save_path=None):
    diff_colors  = {'easy': '#2ecc71', 'medium': '#f39c12', 'hard': '#e74c3c'}
    diff_markers = {'easy': 'o',       'medium': 's',       'hard': 'D'}

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for diff in ['easy', 'medium', 'hard']:
        rows = [r for r in eval_rows if r['difficulty'] == diff]
        if not rows:
            continue
        eps  = [r['episode']      for r in rows]
        succ = [r['success_rate'] for r in rows]
        rew  = [r['reward_mean']  for r in rows]
        ax1.plot(eps, succ, color=diff_colors[diff], marker=diff_markers[diff],
                 linewidth=2, markersize=7, label=diff.capitalize(), zorder=3)
        ax2.plot(eps, rew,  color=diff_colors[diff], marker=diff_markers[diff],
                 linewidth=2, markersize=7, label=diff.capitalize(), zorder=3)

    ax1.axhline(0.5, color='#7f8c8d', linewidth=1, linestyle='--', label='50% line')
    ax1.set_xlabel('Episode', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Success Rate', fontsize=12, fontweight='bold')
    ax1.set_title('Evacuation Success Rate per Difficulty', fontsize=13, fontweight='bold')
    ax1.set_ylim(-0.05, 1.05)
    ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax1.legend(fontsize=10); ax1.grid(True, linestyle='--', linewidth=0.6, alpha=0.7)

    ax2.axhline(0, color='#7f8c8d', linewidth=1, linestyle='--')
    ax2.set_xlabel('Episode', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Mean Reward', fontsize=12, fontweight='bold')
    ax2.set_title('Mean Reward per Difficulty (Eval)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10); ax2.grid(True, linestyle='--', linewidth=0.6, alpha=0.7)

    fig.suptitle(
        f'Pyre PPO — Per-Difficulty Evaluation over {EPISODES} Episodes',
        fontsize=14, fontweight='bold', y=1.02,
    )
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Eval graph saved → {save_path}')
    plt.show(); plt.close(fig)


EVAL_GRAPH_PATH = ARTIFACTS_DIR / 'pyre_ppo_nb_eval.png'
plot_per_difficulty_eval(eval_rows, save_path=EVAL_GRAPH_PATH)
display(IPImage(filename=str(EVAL_GRAPH_PATH)))

## 9 · Learning Improvement — First vs Last Eval

Quantifies **how much the agent improved** on each difficulty over the full run.

In [ ]:
def improvement_summary(eval_rows):
    print('\n' + '=' * 64)
    print(f'{"Difficulty":<10} {"First":>18} {"Last":>18} {"ΔSuccess":>10} {"ΔReward":>10}')
    print('=' * 64)
    for diff in ['easy', 'medium', 'hard']:
        rows = sorted([r for r in eval_rows if r['difficulty'] == diff], key=lambda r: r['episode'])
        if len(rows) < 2:
            continue
        f, l = rows[0], rows[-1]
        ds = l['success_rate'] - f['success_rate']
        dr = l['reward_mean']  - f['reward_mean']
        ts = '↑' if ds > 0.05 else ('↓' if ds < -0.05 else '→')
        tr = '↑' if dr > 0.5  else ('↓' if dr < -0.5  else '→')
        print(
            f'{diff.capitalize():<10}  '
            f'suc={f["success_rate"]:.2f} r={f["reward_mean"]:+.2f}  '
            f'suc={l["success_rate"]:.2f} r={l["reward_mean"]:+.2f}  '
            f'{ts} {ds:+.2f}     {tr} {dr:+.2f}'
        )
    print('=' * 64)


def plot_improvement_bars(eval_rows, save_path=None):
    diffs, firsts, lasts, deltas = [], [], [], []
    for diff in ['easy', 'medium', 'hard']:
        rows = sorted([r for r in eval_rows if r['difficulty'] == diff], key=lambda r: r['episode'])
        if len(rows) < 2:
            continue
        diffs.append(diff.capitalize())
        firsts.append(rows[0]['success_rate'])
        lasts.append(rows[-1]['success_rate'])
        deltas.append(rows[-1]['success_rate'] - rows[0]['success_rate'])

    if not diffs:
        print('Not enough evaluation checkpoints to compare.')
        return

    x       = np.arange(len(diffs))
    width   = 0.35
    colors  = {'Easy': '#2ecc71', 'Medium': '#f39c12', 'Hard': '#e74c3c'}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    bars1 = ax1.bar(x - width/2, firsts, width, label='First Eval',
                    color=[colors[d] for d in diffs], alpha=0.45, edgecolor='black')
    bars2 = ax1.bar(x + width/2, lasts,  width, label='Last Eval',
                    color=[colors[d] for d in diffs], alpha=0.9,  edgecolor='black')
    for bar, v in zip(bars1, firsts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.0%}', ha='center', va='bottom', fontsize=9)
    for bar, v in zip(bars2, lasts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.0%}', ha='center', va='bottom', fontsize=9)
    ax1.set_xticks(x); ax1.set_xticklabels(diffs, fontsize=11)
    ax1.set_ylabel('Success Rate', fontsize=11)
    ax1.set_title('First vs Last Evaluation\nSuccess Rate per Difficulty', fontsize=12, fontweight='bold')
    ax1.set_ylim(0, 1.15)
    ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax1.legend(fontsize=10); ax1.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.7)

    bar_c = ['#27ae60' if d >= 0 else '#c0392b' for d in deltas]
    bars3 = ax2.bar(x, deltas, width=0.5, color=bar_c, edgecolor='black')
    for bar, v in zip(bars3, deltas):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.005 if v >= 0 else -0.025),
                 f'{v:+.0%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2.axhline(0, color='black', linewidth=1)
    ax2.set_xticks(x); ax2.set_xticklabels(diffs, fontsize=11)
    ax2.set_ylabel('Δ Success Rate', fontsize=11)
    ax2.set_title('Learning Improvement\n(Last − First Eval)', fontsize=12, fontweight='bold')
    ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
    ax2.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.7)

    fig.suptitle(f'Pyre PPO — Learning Improvement per Difficulty ({EPISODES} episodes)',
                 fontsize=13, fontweight='bold', y=1.02)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Improvement chart saved → {save_path}')
    plt.show(); plt.close(fig)


improvement_summary(eval_rows)

IMPROV_PATH = ARTIFACTS_DIR / 'pyre_ppo_nb_improvement.png'
plot_improvement_bars(eval_rows, save_path=IMPROV_PATH)
display(IPImage(filename=str(IMPROV_PATH)))

## 10 · Health Drain & Steps Timeline

Shows how agent **survival time (steps)** and **final HP** evolve — indirect indicators
of how efficiently the agent navigates without taking damage.

In [ ]:
def plot_health_and_steps(episode_rows, save_path=None):
    eps   = [r['episode']      for r in episode_rows]
    hp    = [r['final_health'] for r in episode_rows]
    steps = [r['steps']        for r in episode_rows]
    diffs = [r['difficulty']   for r in episode_rows]

    def ma(v, w=20):
        out, run, q = [], 0.0, []
        for x in v:
            q.append(x); run += x
            if len(q) > w: run -= q.pop(0)
            out.append(run / len(q))
        return out

    diff_colors = {'easy': '#d4edda', 'medium': '#fff3cd', 'hard': '#f8d7da'}
    regions = []
    if diffs:
        cur, start = diffs[0], eps[0]
        for ep, d in zip(eps[1:], diffs[1:]):
            if d != cur:
                regions.append((start, ep, cur)); cur, start = d, ep
        regions.append((start, eps[-1], cur))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    for ax in (ax1, ax2):
        for x0, x1, diff in regions:
            ax.axvspan(x0, x1, color=diff_colors.get(diff, '#eee'), alpha=0.35, zorder=0)

    ax1.plot(eps, hp,     color='#bdc3c7', linewidth=0.6, alpha=0.5, label='Final HP')
    ax1.plot(eps, ma(hp), color='#2980b9', linewidth=2.5, label='HP (MA-20)')
    ax1.set_ylabel('Final Health (HP)', fontsize=11, fontweight='bold')
    ax1.set_title('Final Agent Health per Episode', fontsize=12, fontweight='bold')
    ax1.set_ylim(0, 105)

    ax2.plot(eps, steps,     color='#bdc3c7', linewidth=0.6, alpha=0.5, label='Steps')
    ax2.plot(eps, ma(steps), color='#8e44ad', linewidth=2.5, label='Steps (MA-20)')
    ax2.set_xlabel('Episode', fontsize=11, fontweight='bold')
    ax2.set_ylabel('Steps per Episode', fontsize=11, fontweight='bold')
    ax2.set_title('Episode Length over Training', fontsize=12, fontweight='bold')

    patches = [mpatches.Patch(color=diff_colors[d], alpha=0.6, label=d.capitalize())
               for d in ['easy', 'medium', 'hard'] if any(r == d for r in diffs)]
    for ax in (ax1, ax2):
        h, l = ax.get_legend_handles_labels()
        ax.legend(h + patches, l + [p.get_label() for p in patches],
                  fontsize=9, loc='upper right')
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Health/steps chart saved → {save_path}')
    plt.show(); plt.close(fig)


HEALTH_PATH = ARTIFACTS_DIR / 'pyre_ppo_nb_health_steps.png'
plot_health_and_steps(episode_rows, save_path=HEALTH_PATH)
display(IPImage(filename=str(HEALTH_PATH)))

## 11 · Final Summary

In [ ]:
print('=' * 64)
print('  PYRE PPO TRAINING — FINAL RESULTS')
print('=' * 64)
print(f'  Server            : {SERVER_URL}')
print(f'  Device            : {device}')
print(f'  Episodes          : {EPISODES}  (started from {START_EP})')
print(f'  Training time     : {total_time:.1f}s  ({trained_eps/max(1,total_time):.2f} eps/s)')
print(f'  Network params    : {n_params:,}')
print(f'  Architecture      : {HIDDEN_SIZES}')
print(f'  Observation mode  : {OBS_MODE}')
print()
print(f'  Final success rate (last 30 eps) : {float(np.mean(success_window)):.2f}')
print(f'  Final reward mean  (last 30 eps) : {float(np.mean(reward_window)):+.3f}')
print()
print('  Saved artefacts:')
for label, path in [
    ('Model (.pt)',         MODEL_PATH),
    ('Checkpoint (.pt)',    CKPT_PATH),
    ('Training graph',     ARTIFACTS_DIR / 'pyre_ppo_nb_training.png'),
    ('Eval graph',         ARTIFACTS_DIR / 'pyre_ppo_nb_eval.png'),
    ('Improvement chart',  ARTIFACTS_DIR / 'pyre_ppo_nb_improvement.png'),
    ('Health/steps chart', ARTIFACTS_DIR / 'pyre_ppo_nb_health_steps.png'),
]:
    exists = '✓' if Path(path).exists() else '✗'
    print(f'    {exists}  {label:<22} → {path}')
print('=' * 64)